Transformação de dados

In [8]:
import pandas as pd
from camara_deputados.ingestion.data_loader import DataLoader
from camara_deputados.extraction.write import DataWrite
from camara_deputados.transformer.transformer import DataTransformer



In [7]:
# instâncias

bronze = DataLoader('bronze')
salva= DataWrite()
transforma = DataTransformer()



In [26]:
dfs_bronze = bronze.carregar_parquets()


📂 Lendo dados da camada bronze: /mnt/d/GabrielaTorres/estudos/Univesp_Projetos/pi_camara_deputados/notebooks/../data/bronze

✅ deputados_detalhamento: 513 linhas, 34 colunas
✅ deputados_frentes: 131244 linhas, 6 colunas
✅ deputados_lista: 513 linhas, 10 colunas
✅ frentes: 100 linhas, 5 colunas
✅ frentes_detalhamento: 100 linhas, 22 colunas
✅ partidos: 15 linhas, 5 colunas
✅ partidos_detalhamento: 15 linhas, 23 colunas
✅ proposicoes: 60 linhas, 10 colunas
✅ proposicoes_autores: 57 linhas, 8 colunas
✅ proposicoes_detalhamento: 60 linhas, 36 colunas
✅ proposicoes_temas: 62 linhas, 5 colunas
✅ proposicoes_votacoes: 271 linhas, 13 colunas
✅ tipos_proposicao: 544 linhas, 5 colunas
✅ votacoes_detalhamento: 271 linhas, 21 colunas
✅ votacoes_orientacao: 399 linhas, 7 colunas

🎯 Carregamento finalizado.


# Cria camada silver

## Deputado

In [4]:
# criando a silver Deputado

df_deputadoDetalhamento = dfs_bronze['deputados_detalhamento']

colunas_origem = list(df_deputadoDetalhamento)

In [5]:
# as colunas sinalizadas com # não subirão para a silver 

colunas_extraidas = ['id'
    #, 'uri'
    , 'nomeCivil'
    #, 'cpf'
    , 'sexo'
    #, 'urlWebsite'
    #, 'redeSocial'  
    , 'dataNascimento'
    , 'dataFalecimento'
    , 'ufNascimento'
    , 'municipioNascimento'
    , 'escolaridade'
    #, 'ultimoStatus.id'
    #, 'ultimoStatus.uri'
    #, 'ultimoStatus.nome'
    , 'ultimoStatus.siglaPartido'
    #, 'ultimoStatus.uriPartido'
    , 'ultimoStatus.siglaUf'
    , 'ultimoStatus.idLegislatura'
    #, 'ultimoStatus.urlFoto'
    , 'ultimoStatus.email'
    #, 'ultimoStatus.data'
    , 'ultimoStatus.nomeEleitoral'
    #, 'ultimoStatus.gabinete.nome'
    #, 'ultimoStatus.gabinete.sala'
    #, 'ultimoStatus.gabinete.andar'
    #, 'ultimoStatus.gabinete.telefone'
    #, 'ultimoStatus.gabinete.predio'
    #, 'ultimoStatus.gabinete.email'
    , 'ultimoStatus.situacao'
    , 'ultimoStatus.condicaoEleitoral'
    #, 'ultimoStatus.descricaoStatus'
    #, 'source_id'
    #, 'data_extracao'
    ]
df_s_deputado = df_deputadoDetalhamento[colunas_extraidas]

In [6]:
#renomeado as colunas 

mapeamento_deputados = {
    'id':('id_deputado','int')
    ,'nomeCivil':('nom_NomeCivil','str')
    ,'sexo':('nom_Sexo','str')
    ,'dataNascimento':('dat_DataNasc','date')
    ,'dataFalecimento':('dat_DataFalecimento','date')
    ,'ufNascimento':('nom_UFNasc','str')
    ,'municipioNascimento':('nom_MunicipioNasci','str')
    ,'escolaridade':('nom_Escolaridade','str')
    ,'ultimoStatus.siglaPartido':( 'nom_SiglaPartido','str')
    ,'ultimoStatus.siglaUf':('nom_UFRepresenta', 'str')
    ,'ultimoStatus.idLegislatura':('id_Legislatura','int')
    ,'ultimoStatus.email':('nom_Email','str')
    ,'ultimoStatus.nomeEleitoral':('nom_NomeEleitoral','str')
    ,'ultimoStatus.situacao':('nom_Situacao','str')
    ,'ultimoStatus.condicaoEleitoral':('nom_CondEleitoral','str')

}

df_s_deputado = transforma.rename_and_cast(df_s_deputado,mapeamento_deputados)

In [9]:
salva.save_parquet(df_s_deputado, 'silver_deputado', 'silver')

NameError: name 'df_s_deputado' is not defined

## Deputados Frentes

In [10]:
df_deputadoFrentes = dfs_bronze['deputados_frentes']

In [11]:
print(df_deputadoFrentes.columns)

Index(['id', 'uri', 'titulo', 'idLegislatura', 'source_id', 'data_extracao'], dtype='str')


In [13]:
mapeamento_frentes = {
    'id': ('id_Frente', 'int'),
    #'uri': ('des_Uri', 'str'),
    'titulo': ('nom_Titulo', 'str'),
    'idLegislatura': ('id_Legislatura', 'int'),
    'source_id': ('id_Deputado', 'int'),
    #'data_extracao': ('dat_DataExtracao', 'datetime')
}

df_s_frenteDeputado = transforma.rename_and_cast(df_deputadoFrentes,mapeamento_frentes)

In [14]:
df_s_frenteDeputado.sample(5)

,id_Frente,nom_Titulo,id_Legislatura,id_Deputado
82096,54316,Frente Parlamentar de Combate ao Câncer Infantil,57,133810
97422,53687,Frente Parlamentar em Defesa do Serviço Públic...,55,160642
1078,54517,Frente Parlamentar Mista em Defesa dos Saberes...,57,121948
84519,55614,Frente Parlamentar Mista em Defesa e Apoio aos...,57,160535
53023,54345,Frente Parlamentar Parlamentar para a Moderniz...,57,204473


In [15]:
salva.save_parquet(df_s_frenteDeputado,'silver_frenteDeputado', 'silver')

💾 Salvo em: /mnt/d/GabrielaTorres/estudos/Univesp_Projetos/pi_camara_deputados/data/silver/silver_frenteDeputado/silver_frenteDeputado.parquet


## Proposicões Detalhada

In [16]:
#criando a tabela silver das proposições

df_proposicaoDetalhamento = dfs_bronze['proposicoes_detalhamento']

colunas_origem = list(df_proposicaoDetalhamento.columns)
print(colunas_origem)

['id', 'uri', 'siglaTipo', 'codTipo', 'numero', 'ano', 'ementa', 'dataApresentacao', 'uriOrgaoNumerador', 'uriAutores', 'descricaoTipo', 'ementaDetalhada', 'keywords', 'uriPropPrincipal', 'uriPropAnterior', 'uriPropPosterior', 'urlInteiroTeor', 'urnFinal', 'texto', 'justificativa', 'statusProposicao.dataHora', 'statusProposicao.sequencia', 'statusProposicao.siglaOrgao', 'statusProposicao.uriOrgao', 'statusProposicao.uriUltimoRelator', 'statusProposicao.regime', 'statusProposicao.descricaoTramitacao', 'statusProposicao.codTipoTramitacao', 'statusProposicao.descricaoSituacao', 'statusProposicao.codSituacao', 'statusProposicao.despacho', 'statusProposicao.url', 'statusProposicao.ambito', 'statusProposicao.apreciacao', 'source_id', 'data_extracao']


In [22]:
# as colunas sinalizadas com # não subirão para a silver

mapeamento_proposicao= {
    'id':('id_proposicao','int')
    #,'uri'
    ,'siglaTipo':('nom_TipoProposicao','str')
    ,'codTipo':('cod_Tipo','int')
    ,'numero':('num_NumeroProp','int')
    ,'ano':('num_ano', 'int')
    ,'ementa':('nom_Ementa','str')
    ,'dataApresentacao':('dat_Apresentacao','date')
    #,'uriOrgaoNumerador'
    ,'uriAutores':('uri_Autor','str')
    #,'descricaoTipo'
    #,'ementaDetalhada'
    ,'keywords':('nom_Keywords','str')
    #,'uriPropPrincipal'
    #,'uriPropAnterior'
    #,'uriPropPosterior'
    #,'urlInteiroTeor'
    #,'urnFinal'
    #,'texto'
    #,'justificativa'
    #,'statusProposicao.dataHora'
    #,'statusProposicao.sequencia'
    ,'statusProposicao.siglaOrgao':('nom_SiglaOrgao', 'str')
    #,'statusProposicao.uriOrgao'
    ,'statusProposicao.uriUltimoRelator':('uri_Relator', 'str')
    ,'statusProposicao.regime':('nom_regime','str')
    ,'statusProposicao.descricaoTramitacao':('nom_TipoTramitacao','str')
    ,'statusProposicao.codTipoTramitacao':('cod_TipoTramitacao','int')
    ,'statusProposicao.descricaoSituacao':('cod_TipoSiituacao','str')
    ,'statusProposicao.codSituacao':('cod_Situacao','int')
    #,'statusProposicao.despacho'
    ,'statusProposicao.url':('nom_LinkProposicao','str')
    #,'statusProposicao.ambito'
    #,'statusProposicao.apreciacao'
    #,'source_id'
    #,'data_extracao'
}

df_s_proposicao = transforma.rename_and_cast(df_proposicaoDetalhamento, mapeamento_proposicao)

In [23]:
# Pega o id da uri

colunas_uri = {'uri_Autor':('nom_TipoAutor','id_Autor'), 'uri_Relator':('nom_TipoRelator','id_Relator')}

df_s_proposicao = transforma.extrair_tipos_e_ids(df_s_proposicao, colunas_uri)



In [13]:
df_s_proposicao.info()

<class 'pandas.DataFrame'>
RangeIndex: 60 entries, 0 to 59
Data columns (total 19 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   id_proposicao       60 non-null     Int64         
 1   nom_TipoProposicao  60 non-null     string        
 2   cod_Tipo            60 non-null     Int64         
 3   num_NumeroProp      60 non-null     Int64         
 4   num_ano             60 non-null     Int64         
 5   nom_Ementa          60 non-null     string        
 6   dat_Apresentacao    60 non-null     datetime64[us]
 7   uri_Autor           60 non-null     string        
 8   nom_Keywords        40 non-null     string        
 9   nom_SiglaOrgao      58 non-null     string        
 10  uri_Relator         23 non-null     string        
 11  nom_regime          60 non-null     string        
 12  cod_TipoTramitacao  58 non-null     Int64         
 13  cod_Situacao        37 non-null     Int64         
 14  nom_Lin

In [24]:
# salva as proposições na silver

salva.save_parquet(df_s_proposicao, 'silver_proposica','silver')

💾 Salvo em: /mnt/d/GabrielaTorres/estudos/Univesp_Projetos/pi_camara_deputados/data/silver/silver_proposica/silver_proposica.parquet


## Autores Proposicões

In [15]:
df_proposicaoAutores = dfs_bronze['proposicoes_autores']

mapeamento_autores = list(df_proposicaoAutores.columns)

In [16]:
mapeamento_autores = {
    'uri': ('uri_Autor', 'str'),
    'nome': ('nom_Autor', 'str'),
    'codTipo': ('cod_TipoAutor', 'int'),
    'tipo': ('nom_TipoAutor', 'str'),
    'ordemAssinatura': ('num_OrdemAssinatura', 'int'),
    'proponente': ('ind_Proponente', 'str'),  
    'source_id': ('id_proposicao', 'int'),
    'data_extracao': ('dat_Extracao', 'date')
}

df_s_autores  = transforma.rename_and_cast(df_proposicaoAutores, mapeamento_autores)

## Resgata ID da uri

df_s_autores = transforma.extrair_tipos_e_ids(
    df_s_autores,
    {
        'uri_Autor': ('nom_TipoAutor', 'id_Autor')
    }
)

In [19]:
df_s_autores.columns

Index(['uri_Autor', 'nom_Autor', 'cod_TipoAutor', 'nom_TipoAutor',
       'num_OrdemAssinatura', 'ind_Proponente', 'id_proposicao',
       'dat_Extracao', 'id_Autor'],
      dtype='str')

In [18]:
df_s_autores.sample(20)

,uri_Autor,nom_Autor,cod_TipoAutor,nom_TipoAutor,num_OrdemAssinatura,ind_Proponente,id_proposicao,dat_Extracao,id_Autor
29,https://dadosabertos.camara.leg.br/api/v2/orga...,Superior Tribunal de Justiça,50000,orgaos,1,1,2481875,2026-04-19 18:19:55.573900,81
27,https://dadosabertos.camara.leg.br/api/v2/orga...,Poder Executivo,30000,orgaos,1,1,2416826,2026-04-19 18:19:55.573900,253
37,https://dadosabertos.camara.leg.br/api/v2/orga...,Poder Executivo,30000,orgaos,1,1,2481885,2026-04-19 18:19:55.573900,253
44,https://dadosabertos.camara.leg.br/api/v2/orga...,Liderança do Partido Socialista Brasileiro,80000,orgaos,1,1,2599886,2026-04-19 18:19:55.573900,538819
51,https://dadosabertos.camara.leg.br/api/v2/orga...,Poder Executivo,30000,orgaos,1,1,2599895,2026-04-19 18:19:55.573900,253
7,https://dadosabertos.camara.leg.br/api/v2/orga...,Poder Executivo,30000,orgaos,1,1,2345487,2026-04-19 18:19:55.573900,253
13,https://dadosabertos.camara.leg.br/api/v2/orga...,Poder Executivo,30000,orgaos,1,1,2345501,2026-04-19 18:19:55.573900,253
17,https://dadosabertos.camara.leg.br/api/v2/depu...,Sandro Alex,10000,deputados,1,1,2092056,2026-04-19 18:19:55.573900,160621
26,https://dadosabertos.camara.leg.br/api/v2/orga...,Poder Executivo,30000,orgaos,1,1,2416825,2026-04-19 18:19:55.573900,253
28,https://dadosabertos.camara.leg.br/api/v2/orga...,Superior Tribunal de Justiça,50000,orgaos,1,1,2481874,2026-04-19 18:19:55.573900,81


In [22]:
salva.save_parquet(df_s_autores, 'silver_autores','silver')

💾 Salvo em: /mnt/d/GabrielaTorres/estudos/Univesp_Projetos/pi_camara_deputados/data/silver/silver_autores/silver_autores.parquet


## Silver temas das proposicoes

In [24]:
df_proposicoesTemas = dfs_bronze['proposicoes_temas']

list_temas = list(df_proposicoesTemas.columns)

mapeamento_temas = {
    'codTema': ('cod_Tema', 'int'),
    'tema': ('nom_Tema', 'str'),
    'relevancia': ('num_Relevancia', 'int'),
    'source_id': ('id_proposicao', 'int'),
    'data_extracao': ('dat_Extracao', 'date')
}

df_s_temasProposicoes = transforma.rename_and_cast(df_proposicoesTemas,mapeamento_temas)

In [25]:
salva.save_parquet(df_s_temasProposicoes, 'silver_temasProposicao', 'silver')

💾 Salvo em: /mnt/d/GabrielaTorres/estudos/Univesp_Projetos/pi_camara_deputados/data/silver/silver_temasProposicao/silver_temasProposicao.parquet


## Proposições votações

In [40]:
df_proposicoesVotacoes = dfs_bronze['proposicoes_votacoes']

In [27]:
print(list(df_proposicoesVotacoes.columns))

['id', 'uri', 'data', 'dataHoraRegistro', 'siglaOrgao', 'uriOrgao', 'uriEvento', 'proposicaoObjeto', 'uriProposicaoObjeto', 'descricao', 'aprovacao', 'source_id', 'data_extracao']


In [41]:
mapeamento_votacao = {
    'id': ('id_votacao', 'str'),
    #'uri': ('uri_Votacao', 'str'),
    'data': ('dat_DataVotacao', 'date'),
    'dataHoraRegistro': ('dat_DataRegistro', 'date'),
    #'siglaOrgao': ('nom_SiglaOrgao', 'str'),
    'uriOrgao': ('uri_Orgao', 'str'),
    #'uriVotacao': ('uri_votacaoDetalhe', 'str'),
    #'proposicaoObjeto': ('nom_ProposicaoObjeto', 'str'),
    #'uriProposicaoObjeto': ('uri_Proposicao', 'str'),
    'descricao': ('nom_Descricao', 'str'),
    'aprovacao': ('ind_Aprovado', 'str'),  
    'source_id': ('id_proposicao', 'int'),
    #'data_extracao': ('dat_Extracao', 'date')
}

df_s_votacaoProposicao = transforma.rename_and_cast(df_proposicoesVotacoes,mapeamento_votacao)

In [42]:
df_s_votacaoProposicao = transforma.extrair_ids(
    df_s_votacaoProposicao,
    {'uri_Orgao': 'id_Orgao'}
)

In [43]:
df_s_votacaoProposicao.head()

,id_votacao,dat_DataVotacao,dat_DataRegistro,uri_Orgao,nom_Descricao,ind_Aprovado,id_proposicao,id_Orgao
0,369205-154,2023-08-09,2023-08-09 21:59:25,https://dadosabertos.camara.leg.br/api/v2/orga...,Aprovada a Redação Final assinada pela Relator...,1.0,369205,180
1,369205-151,2023-08-09,2023-08-09 21:58:06,https://dadosabertos.camara.leg.br/api/v2/orga...,Aprovada a Subemenda Substitutiva Global ao Pr...,1.0,369205,180
2,369205-129,2023-08-01,2023-08-01 20:17:21,https://dadosabertos.camara.leg.br/api/v2/orga...,Alteração do Regime de Tramitação desta propos...,1.0,369205,180
3,369205-128,2023-08-01,2023-08-01 20:17:21,https://dadosabertos.camara.leg.br/api/v2/orga...,Realizar o encaminhamento do PL-2131/2007 à CI...,1.0,369205,100001
4,369205-127,2023-08-01,2023-08-01 20:17:17,https://dadosabertos.camara.leg.br/api/v2/orga...,Realizar o encaminhamento do PL-2131/2007 à CC...,1.0,369205,100001


In [60]:
salva.save_parquet(df_s_votacaoProposicao, 'silver_votacao_proposicao','silver')

💾 Salvo em: /mnt/d/GabrielaTorres/estudos/Univesp_Projetos/pi_camara_deputados/data/silver/silver_votacao_proposicao/silver_votacao_proposicao.parquet


## Votacões

In [44]:
df_votacaoDetalhamento = dfs_bronze['votacoes_detalhamento']

In [29]:
print(df_votacaoDetalhamento.columns)

Index(['id', 'uri', 'data', 'dataHoraRegistro', 'siglaOrgao', 'uriOrgao',
       'idOrgao', 'uriEvento', 'idEvento', 'descricao', 'aprovacao',
       'descUltimaAberturaVotacao', 'dataHoraUltimaAberturaVotacao',
       'efeitosRegistrados', 'objetosPossiveis', 'proposicoesAfetadas',
       'ultimaApresentacaoProposicao.dataHoraRegistro',
       'ultimaApresentacaoProposicao.descricao',
       'ultimaApresentacaoProposicao.uriProposicaoCitada', 'source_id',
       'data_extracao'],
      dtype='str')


In [48]:
mapeamento_votacaoDetalhamento = {
    'id': ('id_Votacao', 'str'),
    #'uri': ('des_UriVotacao', 'str'),
    'data': ('dat_DataVotacao', 'date'),
    'dataHoraRegistro': ('dat_DataHoraRegistro', 'date'),
    
    'siglaOrgao': ('nom_SiglaOrgao', 'str'),
    #'uriOrgao': ('des_UriOrgao', 'str'),
    'idOrgao': ('id_Orgao', 'int'),
    
    #'uriEvento': ('des_UriEvento', 'str'),
    'idEvento': ('id_Evento', 'int'),
    
    'descricao': ('des_DescricaoVotacao', 'str'),
    'aprovacao': ('ind_Aprovacao', 'str'),
    
    'descUltimaAberturaVotacao': ('des_UltimaAbertura', 'str'),
    'dataHoraUltimaAberturaVotacao': ('dat_UltimaAbertura', 'date'),
    
    'efeitosRegistrados': ('des_Efeitos', 'str'),
    'objetosPossiveis': ('des_Objetos', 'str'),
    'proposicoesAfetadas': ('des_Proposicoes', 'str'),
    
    'ultimaApresentacaoProposicao.dataHoraRegistro': ('dat_UltimaApresentacao', 'date'),
    'ultimaApresentacaoProposicao.descricao': ('des_UltimaApresentacaoDesc', 'str'),
    'ultimaApresentacaoProposicao.uriProposicaoCitada': ('des_UriProposicao', 'str'),
    
    'source_id': ('id_Proposicao', 'int'),
    #'data_extracao': ('dat_DataExtracao', 'date')
}

In [49]:
df_s_votacoesDetalhamento = transforma.rename_and_cast(df_votacaoDetalhamento,mapeamento_votacaoDetalhamento)

df_s_votacoesDetalhamento = transforma.explode_and_extract(df_s_votacoesDetalhamento, 
                                                           'id',
                                                           {'id':'id_ProposicaoAfetada'})

In [52]:
df_s_votacoesDetalhamento.iloc[264]

id_Votacao                                                            2494658-8
dat_DataVotacao                                             2025-04-08 00:00:00
dat_DataHoraRegistro                                        2025-04-08 18:27:25
nom_SiglaOrgao                                                             PLEN
id_Orgao                                                                    180
id_Evento                                                                 75874
des_DescricaoVotacao          Aprovado o Requerimento de Urgência (Art. 155 ...
ind_Aprovacao                                                               1.0
des_UltimaAbertura                                                         <NA>
dat_UltimaAbertura                                                          NaT
des_Efeitos                   [{'dataHoraResultado': '2025-04-08T13:55', 'da...
des_Objetos                                                                  []
des_Proposicoes               [{'ano': 2

In [53]:
df_s_votacoesDetalhamento[df_s_votacoesDetalhamento['id_Votacao']=='2494658-8']

,id_Votacao,dat_DataVotacao,dat_DataHoraRegistro,nom_SiglaOrgao,id_Orgao,id_Evento,des_DescricaoVotacao,ind_Aprovacao,des_UltimaAbertura,dat_UltimaAbertura,des_Efeitos,des_Objetos,des_Proposicoes,dat_UltimaApresentacao,des_UltimaApresentacaoDesc,des_UriProposicao,id_Proposicao
264,2494658-8,2025-04-08,2025-04-08 18:27:25,PLEN,180,75874,Aprovado o Requerimento de Urgência (Art. 155 ...,1.0,<NA>,NaT,"[{'dataHoraResultado': '2025-04-08T13:55', 'da...",[],"[{'ano': 2025, 'codTipo': 139, 'dataApresentac...",NaT,<NA>,<NA>,<NA>


In [59]:
salva.save_parquet(df_votacaoDetalhamento, 'silver_votacao_detalhamento','silver')

💾 Salvo em: /mnt/d/GabrielaTorres/estudos/Univesp_Projetos/pi_camara_deputados/data/silver/silver_votacao_detalhamento/silver_votacao_detalhamento.parquet


## Votações Orientação

In [54]:
df_votacoesOrientacao = dfs_bronze['votacoes_orientacao']

In [56]:
print(df_votacoesOrientacao.columns)

Index(['orientacaoVoto', 'codTipoLideranca', 'siglaPartidoBloco',
       'codPartidoBloco', 'uriPartidoBloco', 'source_id', 'data_extracao'],
      dtype='str')


In [57]:
mapeamento_orientacaoVoto = {
    'orientacaoVoto': ('nom_OrientacaoVoto', 'str'),
    'codTipoLideranca': ('id_TipoLideranca', 'int'),
    
    'siglaPartidoBloco': ('nom_SiglaPartidoBloco', 'str'),
    'codPartidoBloco': ('id_PartidoBloco', 'int'),
    'uriPartidoBloco': ('des_UriPartidoBloco', 'str'),
    
    'source_id': ('id_Votacao', 'str'),
    'data_extracao': ('dat_DataExtracao', 'date')
}

df_s_votosOrientacao = transforma.rename_and_cast(df_votacoesOrientacao,
                                                mapeamento_orientacaoVoto)

In [62]:
df_s_votacaoProposicao.head(10)

,id_votacao,dat_DataVotacao,dat_DataRegistro,uri_Orgao,nom_Descricao,ind_Aprovado,id_proposicao,id_Orgao,data_extracao
0,369205-154,2023-08-09,2023-08-09 21:59:25,https://dadosabertos.camara.leg.br/api/v2/orga...,Aprovada a Redação Final assinada pela Relator...,1.0,369205,180,2026-04-22 07:08:08.069286
1,369205-151,2023-08-09,2023-08-09 21:58:06,https://dadosabertos.camara.leg.br/api/v2/orga...,Aprovada a Subemenda Substitutiva Global ao Pr...,1.0,369205,180,2026-04-22 07:08:08.069286
2,369205-129,2023-08-01,2023-08-01 20:17:21,https://dadosabertos.camara.leg.br/api/v2/orga...,Alteração do Regime de Tramitação desta propos...,1.0,369205,180,2026-04-22 07:08:08.069286
3,369205-128,2023-08-01,2023-08-01 20:17:21,https://dadosabertos.camara.leg.br/api/v2/orga...,Realizar o encaminhamento do PL-2131/2007 à CI...,1.0,369205,100001,2026-04-22 07:08:08.069286
4,369205-127,2023-08-01,2023-08-01 20:17:17,https://dadosabertos.camara.leg.br/api/v2/orga...,Realizar o encaminhamento do PL-2131/2007 à CC...,1.0,369205,100001,2026-04-22 07:08:08.069286
5,369205-126,2023-08-01,2023-08-01 20:17:12,https://dadosabertos.camara.leg.br/api/v2/orga...,Realizar o encaminhamento do PL-2131/2007 à CC...,1.0,369205,100001,2026-04-22 07:08:08.069286
6,369205-71,2013-04-10,2013-04-10 12:18:24,https://dadosabertos.camara.leg.br/api/v2/orga...,Aprovado por Unanimidade o Parecer.,1.0,369205,2010,2026-04-22 07:08:08.069286
7,369205-38,2009-10-21,2009-10-21 11:32:33,https://dadosabertos.camara.leg.br/api/v2/orga...,Aprovado por Unanimidade o Parecer.,1.0,369205,2014,2026-04-22 07:08:08.069286
8,618609-107,2025-08-27,2025-08-27 14:12:00,https://dadosabertos.camara.leg.br/api/v2/orga...,Aprovado o Parecer.,1.0,618609,536996,2026-04-22 07:08:08.069286
9,618609-80,2023-12-12,2023-12-12 14:58:47,https://dadosabertos.camara.leg.br/api/v2/orga...,Aprovada a Redação Final.,1.0,618609,2003,2026-04-22 07:08:08.069286


Criando as dimensões

In [55]:
# criando a dim_deputados
df_deputadoDetalhamento = bronze.carregar_tabela('deputados_detalhamento')

# Mapeia as colunas que irão para camada silver e nomeia
mapeamento = {'id':'id_deputado',
           'nomeCivil':'nom_Nome',
           'sexo':'nom_Sexo',
           'dataNascimento':'dat_DataNascimento',
           'ufNascimento':'nom_UF',
           'municipioNascimento':'nom_MunicipioNatal',
           'escolaridade':'nom_Escolaridade'
}

#Resgata as colunas originais
colunas_origem = list(mapeamento.keys())


#cria a  dim_deputado com as colunas renomeadas
dim_deputado = df_deputadoDetalhamento[colunas_origem].rename(columns=mapeamento)



✅ deputados_detalhamento: 513 linhas, 34 colunas


In [56]:
dim_deputado.sample(5)

,id_deputado,nom_Nome,nom_Sexo,dat_DataNascimento,nom_UF,nom_MunicipioNatal,nom_Escolaridade
466,204557,SIDNEY RICARDO DE OLIVEIRA LEITE,M,1967-04-08,AM,Manaus,Superior
423,73801,RENILDO VASCONCELOS CALHEIROS,M,1959-04-20,AL,Murici,Superior
376,220706,NELSON FERNANDO PADOVANI,M,1977-10-29,PR,Cascavel,Superior
259,214694,JORGE GOETTEN DE LIMA,M,1962-04-10,SC,Mirim Doce,Superior
141,160599,DIMAS FABIANO TOLEDO JÚNIOR,M,1973-05-23,RJ,Macaé,Superior


In [57]:
dim_deputado['dat_DataNascimento']=pd.to_datetime(dim_deputado['dat_DataNascimento'],errors='coerce')

dim_deputado = dim_deputado.sort_values("id_deputado").reset_index(drop=True)



In [58]:
# salvando na silver

salva.save_parquet(dim_deputado, 'dim_s_deputados', layer='gold_temp')

💾 Salvo em: /mnt/d/GabrielaTorres/estudos/Univesp_Projetos/pi_camara_deputados/data/gold_temp/dim_s_deputados/dim_s_deputados.parquet


In [59]:
## Dim_frentes


df_frentes = bronze.carregar_tabela('frentes')

mapeamento_frentes = {'id':'id_frentes',
                      'titulo':'nom_TituloFrente',
                      'idLegislatura':'num_Legislatura'

}

colunas_origem = list(mapeamento_frentes.keys())

dim_frente = df_frentes[colunas_origem].rename(columns=mapeamento_frentes)

✅ frentes: 100 linhas, 5 colunas


In [60]:
dim_frente.sample(5)

,id_frentes,nom_TituloFrente,num_Legislatura
91,55570,Frente Parlamentar do Desporto Escolar,57
88,54563,"Frente Parlamentar em Apoio ao Petróleo, Gás e...",57
85,55568,Frente Parlamentar Mista da Medicina,57
4,55710,Frente Parlamentar Mista Brasil - Espanha,57
78,54560,"Frente Parlamentar Mista em Defesa da Criança,...",57


In [61]:
# salva na camada gold_temp 

salva.save_parquet(dim_frente, 'dim_s_frente', 'gold_temp')

💾 Salvo em: /mnt/d/GabrielaTorres/estudos/Univesp_Projetos/pi_camara_deputados/data/gold_temp/dim_s_frente/dim_s_frente.parquet


In [62]:
# Cria dim_partido na gold_temp

df_partidos = bronze.carregar_tabela('partidos')

mapeamento_partidos = {'id':'id_partido',
                       'sigla':'nom_Sigla',
                       'nome':'nom_NomePartido'}

colunas_origem = list(mapeamento_partidos.keys())

dim_partido = df_partidos[colunas_origem].rename(columns=mapeamento_partidos)

✅ partidos: 15 linhas, 5 colunas


In [63]:
dim_partido.sample(5)

,id_partido,nom_Sigla,nom_NomePartido
14,36839,PSOL,Partido Socialismo e Liberdade
9,37903,PP,Progressistas
2,36899,MDB,Movimento Democrático Brasileiro
4,37901,NOVO,Partido Novo
6,36786,PDT,Partido Democrático Trabalhista


In [64]:
salva.save_parquet(dim_partido,'dim_s_partido','gold_temp')

💾 Salvo em: /mnt/d/GabrielaTorres/estudos/Univesp_Projetos/pi_camara_deputados/data/gold_temp/dim_s_partido/dim_s_partido.parquet


In [65]:
# Criando a dim_proposicao

df_proposicao = bronze.carregar_tabela('proposicoes_detalhamento')

mapeamento_proposicao = {
    'id':'id_proposicao',
    'codTipo':'cod_Tipo',
    'numero':'num_NumeroProposicao',
    'ano':'num_Ano',
    'ementa':'nom_Ementa',
    'keywords':'nom_Keywords'
}

colunas_origem = list(mapeamento_proposicao.keys())

dim_proposicao = df_proposicao[colunas_origem].rename(columns = mapeamento_proposicao)

✅ proposicoes_detalhamento: 60 linhas, 36 colunas


In [67]:
salva.save_parquet(dim_proposicao,'dim_s_proposicao', 'gold_temp')

💾 Salvo em: /mnt/d/GabrielaTorres/estudos/Univesp_Projetos/pi_camara_deputados/data/gold_temp/dim_s_proposicao/dim_s_proposicao.parquet


Criando a dim_tema a partir da tabela proposicoes Temas

In [ ]:
#Carregando a proposicoe_temas

df_tema = bronze.carregar_tabela('proposicoes_temas')

✅ proposicoes_temas: 62 linhas, 5 colunas


In [68]:
dim_tema = (
    df_tema[['codTema', 'tema']]
    .drop_duplicates()
    .dropna(subset=['codTema'])
    .sort_values('codTema')
    .reset_index(drop=True)
)

In [69]:
display(dim_tema)

,codTema,tema
0,34,Administração Pública
1,37,Comunicações
2,40,Economia
3,41,Cidades e Desenvolvimento Urbano
4,42,Direito Civil e Processual Civil
5,43,Direito Penal e Processual Penal
6,44,Direitos Humanos e Minorias
7,46,Educação
8,48,Meio Ambiente e Desenvolvimento Sustentável
9,52,Previdência e Assistência Social


In [70]:
# renomeando e salvando na silver 
 
mapeamento_temas = {
    'codTema':'cod_Tema',
    'tema':'nom_Tema'
}

colunas_origem = list(mapeamento_temas.keys())

dim_tema = dim_tema[colunas_origem].rename(columns=mapeamento_temas)

salva.save_parquet(dim_tema, 'dim_s_tema', 'gold_temp')

💾 Salvo em: /mnt/d/GabrielaTorres/estudos/Univesp_Projetos/pi_camara_deputados/data/gold_temp/dim_s_tema/dim_s_tema.parquet


In [71]:
def analisar_dfs(dfs):
    resultado = []

    for nome_tabela, df in dfs.items():
        for coluna in df.columns:
            
            total = len(df)
            nulos = df[coluna].isnull().sum()
            
            resultado.append({
                "tabela": nome_tabela,
                "coluna": coluna,
                "tipo": df[coluna].dtype,
                "nulos": nulos,
                "%_nulos": nulos / total,
                "unicos": df[coluna].astype(str).nunique(),
                "total_linhas": total
            })

    return pd.DataFrame(resultado)

In [72]:
dfs_dim = {
    'dim_deputado':dim_deputado,
    'dim_frente':dim_frente,
    'dim_partido':dim_partido,
    'dim_proposicacao':dim_proposicao,
    'dim_tema':dim_tema

}

In [73]:
df_analiseDim = analisar_dfs(dfs_dim)

In [74]:
display(df_analiseDim)

,tabela,coluna,tipo,nulos,%_nulos,unicos,total_linhas
0,dim_deputado,id_deputado,int64,0,0.000000,513,513
1,dim_deputado,nom_Nome,str,0,0.000000,513,513
2,dim_deputado,nom_Sexo,str,0,0.000000,2,513
3,dim_deputado,dat_DataNascimento,datetime64[us],0,0.000000,502,513
4,dim_deputado,nom_UF,str,1,0.001949,27,513
5,dim_deputado,nom_MunicipioNatal,str,2,0.003899,267,513
6,dim_deputado,nom_Escolaridade,str,12,0.023392,11,513
7,dim_deputado,data_extracao,datetime64[us],0,0.000000,1,513
8,dim_frente,id_frentes,int64,0,0.000000,100,100
9,dim_frente,nom_TituloFrente,str,0,0.000000,100,100
